# Estudo de caso 2.2 - Diferença salarial entre homens e mulheres

Configuração do *notebook*:

Sincronize sua conta do Google. Para isso, siga o link que aparece na saída da seguinte célula uma vez executada. Copie o código que aparece na tela e insira-o na saída da célula. Assim que visualizar a mensagem: `Google Drive sincronizado com sucesso!` poderá continuar executando o restante das células.

In [ ]:
!pip install rpy2==3.5.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.7/201.7 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for rpy2: filename=rpy2-3.5.1-cp310-cp310-linux_x86_64.whl size=318099 sha256=8bcf6918811c397795bf724671688f8e3d200e3ead3bf137162917a404e2d0f8
  Stored in directory: /root/.cache/pip/wheels/73/a6/ff/4e75dd1ce1cfa2b9a670cbccf6a1e41c553199e9b25f05d953
Successfully built rpy2
  Attempting uninstall: rpy2
    Found existing installation: rpy2 3.5.5
    Uninstalling rpy2-3.5.5:
      Successfully uninstalled rpy2-3.5.5


In [ ]:
from google.colab import auth
auth.authenticate_user()

from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)
data_drop = drive.CreateFile({'id':'1U5UQdm43w9zh2w1VDxevQsb-vp3T7YCj'})
data_drop.GetContentFile('pay.discrimination.Rdata')

print('Google Drive sincronizado com sucesso!')

Google Drive sincronizado com sucesso!


In [ ]:
%load_ext rpy2.ipython

## Dados


In [ ]:
%%R
# Carregar o banco de dados
load(file="pay.discrimination.Rdata")

# Mostrar as variáveis do banco de dados
class(data)
str(data)

# Anexar o banco de dados ao espaço de trabalho
attach(data)

# Mostrar dimensões do banco de dados
dims  <- dim(data)
cat('\nDimensões do banco de dados:',toString(dims),'\n',fill = TRUE)

'data.frame':	3835 obs. of  12 variables:
 $ female: num  0 0 0 0 1 0 1 0 0 0 ...
 $ cg    : num  0 1 0 1 1 0 0 0 0 0 ...
 $ sc    : num  0 0 1 0 0 1 1 0 0 1 ...
 $ hsg   : num  1 0 0 0 0 0 0 1 1 0 ...
 $ mw    : num  0 0 0 0 0 0 0 0 0 0 ...
 $ so    : num  0 0 0 0 0 0 0 0 0 0 ...
 $ we    : num  0 0 0 0 0 0 0 0 0 0 ...
 $ ne    : num  1 1 1 1 1 1 1 1 1 1 ...
 $ exp1  : num  33 27 13 2 15 6.5 6 25 14 26 ...
 $ exp2  : num  10.89 7.29 1.69 0.04 2.25 ...
 $ exp3  : num  35.937 19.683 2.197 0.008 3.375 ...
 $ wage  : num  11.66 12.82 5.78 12.47 18.52 ...

Dimensões do banco de dados: 3835, 12 



Elaboração de uma tabela com a média de cada variável, controlando por gênero:

In [ ]:
%%R

stats.female  <- as.matrix(apply(data[female==1,], 2, mean))
stats.male    <- as.matrix(apply(data[female==0,], 2, mean))
stats         <- cbind(stats.male, stats.female)

colnames(stats) = c("média homens", "média mulheres")
print(stats,digits=2)

       média homens média mulheres
female         0.00           1.00
cg             0.35           0.41
sc             0.30           0.35
hsg            0.34           0.24
mw             0.28           0.29
so             0.24           0.26
we             0.22           0.20
ne             0.26           0.26
exp1          13.58          13.04
exp2           2.59           2.45
exp3           5.96           5.60
wage          16.12          14.72


## Metodologia

### Regressão linear

Modelo básico:

In [ ]:
%%R
# Regressão linear do salário
fmla1     <- wage ~  female + sc+ cg+ mw + so + we + exp1 + exp2 + exp3

# Execução de OLS, obter coeficientes, erros padrão e intervalos de confiança de 95%
full.fit1 <- lm(fmla1, data=data)
est1      <- summary(full.fit1)$coef[2,1:2]
ci1       <-  confint(full.fit1)[2,]

Modelo flexível:

In [ ]:
%%R
# Regressão linear: especificação quadrática
fmla2     <-  wage ~  female + (sc+ cg+ mw + so + we + exp1 + exp2 + exp3)^2

# Execução de OLS, obter coeficientes, erros padrão e intervalos de confiança de 95%
full.fit2 <- lm(fmla2, data=data)
est2      <- summary(full.fit2)$coef[2,1:2]
ci2       <- confint(full.fit2)[2,]

In [ ]:
%%R
# Resumo das especificações linear e quadrática
table1     <- matrix(0, 2, 4)
table1[1,] <- c(est1,ci1)
table1[2,] <- c(est2,ci2)

# Atribuindo nomes a filas e columnas
colnames(table1) <- c("Estimativa (beta)", "Erro padrão", "Inter. Conf. Inf.", "Inter. Conf. Sup.")
rownames(table1) <- c("reg básica", "reg flexível")

### Particionamento

Especificação linear (modelo básico)

In [ ]:
%%R
# Regressão linear de y (resultado) nas covariáveis
fmla1.y <- wage ~  sc+ cg+ mw + so + we + exp1 + exp2 + exp3

# Regressão linear de d (regressor objetivo/tratamento) nas covariáveis
fmla1.d <- female ~  sc+ cg+ mw + so + we + exp1 + exp2 + exp3

# Residuais da regressão de y
t.Y    <- lm(fmla1.y, data=data)$res

# Residuais da regressão de d
t.D    <-  lm(fmla1.d, data=data)$res


# Execução de OLS, obter coeficientes, erros padrão e intervalos de confiança de 95%
partial.fit1   <- lm(t.Y~t.D)
partial.est1   <- summary(partial.fit1)$coef[2,1:2]
partial.ci1    <- confint(partial.fit1)[2,]

Especificação quadrática (modelo flexível)

In [ ]:
%%R
fmla2.y  <- wage ~  (sc+ cg+ mw + so + we + exp1 + exp2 + exp3)^2
fmla2.d  <- female ~ (sc+ cg+ mw + so + we + exp1 + exp2 + exp3)^2

# Obter residuais da regressão linear
t.Y  <- lm(fmla2.y, data=data)$res
t.D  <- lm(fmla2.d, data=data)$res

# Regressão dos residuais entre si para obter o resultado do particionamento
partial.fit2  <-  lm(t.Y~t.D)
partial.est2  <-  summary(partial.fit2)$coef[2,1:2]
partial.ci2   <-  confint(partial.fit2)[2,]


# Criar tabela para coletar os resultados
table2     <- matrix(0, 4, 2)
table2[1,] <- c(est1)
table2[2,] <- c(est2)
table2[3,] <- c(partial.est1)
table2[4,] <- c(partial.est2)

# Atribuir nomes de filas e colunas
colnames(table2) <- c("Estimativa (beta)", "Erro padrão")
rownames(table2) <- c("reg básica", "reg flexível", "reg básica com particionamento", "reg flexível com particionamento")

## Resultados

In [ ]:
%%R
# Mostrar resultados
cat('- Regressão linear:\n',fill = TRUE)
print(table1,digits=3)

cat('\n\n- Comparação com e sem particionamento:\n',fill = TRUE)
print(table2,digits=3)

- Regressão linear:

             Estimativa (beta) Erro padrão Inter. Conf. Inf. Inter. Conf. Sup.
reg básica               -1.83       0.425             -2.66            -0.994
reg flexível             -1.88       0.425             -2.71            -1.047


- Comparação com e sem particionamento:

                                 Estimativa (beta) Erro padrão
reg básica                                   -1.83       0.425
reg flexível                                 -1.88       0.425
reg básica com particionamento               -1.83       0.424
reg flexível com particionamento             -1.88       0.423
